# FlowCLIP - Optical-Flow Encoding Comparison (proof-of-concept)

Compares how the choice of **optical-flow encoding** affects action recognition, using a
**frozen CLIP ViT-B/16** backbone on **HMDB51**. Every encoding is evaluated as a
**linear probe** on fixed CLIP features, so differences reflect the *encoding*, not the
training budget. This answers Vidya's question: *which flow encoding, and why?*

**Encodings compared**
1. `rgb`           - RGB frames (baseline)
2. `flow_xy`       - flow stuffed into 3 channels `[flow_x, flow_y, magnitude]` (the naive "2->3" approach FlowCLIP uses)
3. `flow_hsv`      - flow rendered as an RGB image via HSV / Middlebury colour-coding (angle->hue, magnitude->value) **[literature-recommended]**
4. `flow_temporal` - temporal stack: flow magnitude at start/mid/end packed into R/G/B (keeps motion over time)

Plus **late fusion** `rgb + <flow>`.

> **Hypothesis under test** (from a literature review): `flow_hsv` should beat the naive `flow_xy`, because
> flow's value is its *appearance-invariance* (which colour-coding preserves) and a 3-channel colour image
> matches the RGB distribution the frozen CLIP encoder was pretrained on, whereas a raw 2->3 projection is
> out-of-distribution for a frozen encoder. No prior paper has benchmarked this on a frozen CLIP ViT - this
> notebook is what tests it.

**Runtime:** Runtime -> Change runtime type -> **GPU** (T4/L4 fine). ~1-3 hrs depending on `MAX_PER_CLASS`.

> Proof-of-concept note: uses a fixed per-class train/test split (seed=1), **not** the official 3-split
> protocol, and **Farneback** flow by default for speed. Both are flagged and easy to swap (config cell).
> The comparison across encodings is fair (same split, same flow).


In [ ]:
# Cell 1 - GPU check
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU - set Runtime > Change runtime type > GPU')

In [ ]:
# Cell 2 - Config
CONFIG = dict(
    DATASET_ROOT   = '/content/hmdb51',
    CACHE_DIR      = '/content/feat_cache',
    N_FRAMES       = 8,        # RGB frames sampled per video
    MAX_PER_CLASS  = 40,       # cap videos/class to bound runtime (None = all 6766 videos)
    TRAIN_FRAC     = 0.7,      # per-class train fraction (proof-of-concept split)
    SEED           = 1,
    FLOW_METHOD    = 'farneback',  # 'farneback' (fast) or 'tvl1' (sharper, slower; needs opencv-contrib)
    CLIP_ARCH      = 'ViT-B/16',
    DECODE_SIZE    = 256,      # resize short side before flow (speed)
    BATCH          = 256,      # CLIP forward batch size
)
import pprint; pprint.pprint(CONFIG)

In [ ]:
# Cell 3 - Install deps
import importlib, subprocess, sys
def ensure(imp, pip=None):
    try: importlib.import_module(imp); print(imp, 'ok')
    except Exception:
        subprocess.check_call([sys.executable,'-m','pip','install','-q', pip or imp]); print('installed', pip or imp)
ensure('cv2', 'opencv-contrib-python-headless')   # contrib -> has cv2.optflow (TV-L1)
ensure('decord')
ensure('sklearn', 'scikit-learn')
ensure('tqdm'); ensure('pandas')
try:
    import clip
except Exception:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','git+https://github.com/openai/CLIP.git'])
import clip, cv2, torch
print('CLIP ok | OpenCV', cv2.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print('TV-L1 available:', hasattr(cv2, 'optflow') and hasattr(cv2.optflow, 'DualTVL1OpticalFlow_create'))

In [ ]:
# Cell 4 - Download HMDB51 videos from the HF mirror
import os, glob, zipfile, shutil, subprocess
ROOT = CONFIG['DATASET_ROOT']; VID = ROOT + '/videos'
os.makedirs(VID, exist_ok=True)
if len([d for d in glob.glob(VID + '/*') if os.path.isdir(d)]) < 51:
    subprocess.run(['pip','-q','install','huggingface_hub'], check=False)
    from huggingface_hub import snapshot_download
    local = snapshot_download(repo_id='jili5044/hmdb51', repo_type='dataset', local_dir=ROOT + '/hf')
    zpath = ROOT + '/hf/hmdb51.zip'
    print('extracting', zpath)
    with zipfile.ZipFile(zpath) as zf: zf.extractall(ROOT + '/extracted')
    class_dirs = set(os.path.dirname(p) for p in glob.glob(ROOT + '/extracted/**/*.avi', recursive=True))
    for cd in class_dirs:
        dst = os.path.join(VID, os.path.basename(cd))
        if not os.path.exists(dst): shutil.move(cd, dst)
n_cls = len([d for d in os.listdir(VID) if os.path.isdir(VID + '/' + d)])
n_vid = len(glob.glob(VID + '/**/*.avi', recursive=True))
print('HMDB51:', n_cls, 'classes,', n_vid, 'videos')

In [ ]:
# Cell 5 - Build the (proof-of-concept) train/test split
import os, glob, random
random.seed(CONFIG['SEED'])
VID = CONFIG['DATASET_ROOT'] + '/videos'
classes = sorted([d for d in os.listdir(VID) if os.path.isdir(VID + '/' + d)])
cls2idx = {c:i for i,c in enumerate(classes)}
train, test = [], []
for c in classes:
    vids = sorted(glob.glob(VID + '/' + c + '/*.avi'))
    random.shuffle(vids)
    if CONFIG['MAX_PER_CLASS']: vids = vids[:CONFIG['MAX_PER_CLASS']]
    k = int(len(vids)*CONFIG['TRAIN_FRAC'])
    train += [(v, cls2idx[c]) for v in vids[:k]]
    test  += [(v, cls2idx[c]) for v in vids[k:]]
print(len(classes), 'classes | train', len(train), '| test', len(test))

In [ ]:
# Cell 6 - Flow + encoding utilities
import cv2, numpy as np

_tvl1 = None
def _flow(prev, cur):
    global _tvl1
    if CONFIG['FLOW_METHOD'] == 'tvl1' and hasattr(cv2,'optflow'):
        if _tvl1 is None: _tvl1 = cv2.optflow.DualTVL1OpticalFlow_create()
        return _tvl1.calc(prev, cur, None)
    return cv2.calcOpticalFlowFarneback(prev, cur, None, 0.5,3,15,3,5,1.2,0)

def _norm8(a):
    a = a.astype(np.float32); a -= a.min(); m = a.max()
    return (a/m*255 if m>0 else a*0).astype(np.uint8)

def enc_flow_xy(flow):                     # [flow_x, flow_y, magnitude]  (naive 2->3)
    fx, fy = flow[...,0], flow[...,1]
    mag = np.sqrt(fx*fx + fy*fy)
    return np.stack([_norm8(fx), _norm8(fy), _norm8(mag)], -1)

def enc_flow_hsv(flow):                     # HSV / Middlebury colour-coding -> RGB
    mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])
    hsv = np.zeros((flow.shape[0], flow.shape[1], 3), np.uint8)
    hsv[...,0] = (ang * 180/np.pi/2).astype(np.uint8)
    hsv[...,1] = 255
    hsv[...,2] = cv2.normalize(mag, None, 0,255, cv2.NORM_MINMAX).astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

def enc_flow_temporal(flows):               # magnitude at start/mid/end -> R/G/B
    idx = [0, len(flows)//2, len(flows)-1]
    chans = []
    for i in idx:
        f = flows[i]; mag = np.sqrt(f[...,0]**2 + f[...,1]**2)
        chans.append(cv2.normalize(mag, None, 0,255, cv2.NORM_MINMAX).astype(np.uint8))
    return np.stack(chans, -1)

def read_video(path, n, size):
    import decord
    vr = decord.VideoReader(path, num_threads=1)
    L = len(vr)
    if L < 2: return None
    idx = np.linspace(0, L-1, n, dtype=int)
    frames = vr.get_batch(list(idx)).asnumpy()   # n,H,W,3 RGB
    h, w = frames.shape[1], frames.shape[2]; s = size/min(h,w)
    return np.stack([cv2.resize(f, (int(w*s), int(h*s))) for f in frames])

print('flow / encoding utils ready')

In [ ]:
# Cell 7 - Per-video feature extraction with frozen CLIP
import torch, clip, numpy as np, os
from PIL import Image
from tqdm import tqdm

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
model, preprocess = clip.load(CONFIG['CLIP_ARCH'], device=DEV, jit=False)
model.eval()
for p in model.parameters(): p.requires_grad_(False)

@torch.no_grad()
def clip_embed(imgs_uint8):                 # list of HxWx3 uint8 RGB -> (512,) mean-pooled, L2
    if not imgs_uint8: return np.zeros(512, np.float32)
    batch = torch.stack([preprocess(Image.fromarray(im)) for im in imgs_uint8]).to(DEV)
    feats = []
    for i in range(0, len(batch), CONFIG['BATCH']):
        feats.append(model.encode_image(batch[i:i+CONFIG['BATCH']]).float().cpu())
    f = torch.cat(feats); f = f / f.norm(dim=-1, keepdim=True)
    return f.mean(0).numpy()

ENCODINGS = ['rgb', 'flow_xy', 'flow_hsv', 'flow_temporal']

def video_features(path):
    fr = read_video(path, CONFIG['N_FRAMES'], CONFIG['DECODE_SIZE'])
    if fr is None: return None
    gray = [cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in fr]
    flows = [_flow(gray[i], gray[i+1]) for i in range(len(gray)-1)]
    out = {}
    out['rgb']           = clip_embed(list(fr))
    out['flow_xy']       = clip_embed([enc_flow_xy(f)  for f in flows])
    out['flow_hsv']      = clip_embed([enc_flow_hsv(f) for f in flows])
    out['flow_temporal'] = clip_embed([enc_flow_temporal(flows)]) if flows else np.zeros(512, np.float32)
    return out

def extract(split, name):
    os.makedirs(CONFIG['CACHE_DIR'], exist_ok=True)
    fpath = CONFIG['CACHE_DIR'] + '/' + name + '.npz'
    if os.path.exists(fpath):
        d = dict(np.load(fpath)); print('cached', name, d['y'].shape); return d
    feats = {e: [] for e in ENCODINGS}; ys = []
    for path, y in tqdm(split, desc=name):
        try: vf = video_features(path)
        except Exception: vf = None
        if vf is None: continue
        for e in ENCODINGS: feats[e].append(vf[e])
        ys.append(y)
    arr = {e: np.stack(feats[e]) for e in ENCODINGS}; arr['y'] = np.array(ys)
    np.savez(fpath, **arr); print('saved', name, arr['y'].shape)
    return arr

train_f = extract(train, 'train')
test_f  = extract(test,  'test')

In [ ]:
# Cell 8 - Linear probes + late fusion -> results table
import numpy as np, pandas as pd, json
from sklearn.linear_model import LogisticRegression

def l2(x): return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-8)

def probe(Xtr, ytr, Xte, yte):
    clf = LogisticRegression(max_iter=2000, C=1.0, n_jobs=-1)
    clf.fit(l2(Xtr), ytr)
    return 100.0 * clf.score(l2(Xte), yte)

rows = []
rgb_acc = probe(train_f['rgb'], train_f['y'], test_f['rgb'], test_f['y'])
rows.append(dict(encoding='rgb (baseline)', flow_only=float('nan'), rgb_plus_flow=rgb_acc))

for e in ['flow_xy', 'flow_hsv', 'flow_temporal']:
    flow_only = probe(train_f[e], train_f['y'], test_f[e], test_f['y'])
    Xtr = np.concatenate([l2(train_f['rgb']), l2(train_f[e])], 1)
    Xte = np.concatenate([l2(test_f['rgb']),  l2(test_f[e])],  1)
    fused = probe(Xtr, train_f['y'], Xte, test_f['y'])
    rows.append(dict(encoding=e, flow_only=flow_only, rgb_plus_flow=fused))

df = pd.DataFrame(rows).round(2)
print(df.to_string(index=False))
df.to_csv(CONFIG['CACHE_DIR'] + '/results.csv', index=False)

summary = dict(config={k:v for k,v in CONFIG.items()},
               n_train=int(len(train_f['y'])), n_test=int(len(test_f['y'])),
               n_classes=int(len(set(test_f['y'].tolist()))), results=rows)
with open(CONFIG['CACHE_DIR'] + '/results.json','w') as f: json.dump(summary, f, indent=2, default=float)
print('\nSaved results.csv and results.json in', CONFIG['CACHE_DIR'])
print('>>> Paste the table above (or upload results.json) back to Claude to finalise the Vidya email.')

## How to read this

- **`flow_only`** = linear probe on that flow encoding alone. **`rgb_plus_flow`** = probe on RGB+flow concatenated (late fusion).
- The encoding whose `rgb_plus_flow` most exceeds the **rgb baseline** (and whose `flow_only` is highest) wins.
- The key contrast is **`flow_xy`** (FlowCLIP's current naive approach) vs **`flow_hsv`** (literature-recommended) vs **`flow_temporal`**.

**Send back:** copy the printed table, or download `feat_cache/results.json`, and paste it to Claude - it drops the
numbers into the Vidya email and locks the recommendation. To move toward publishable numbers later:
`MAX_PER_CLASS=None`, `FLOW_METHOD='tvl1'`, and swap the random split for the official HMDB51 3-split files
(then report the 3-split mean, add UCF101).
